In [7]:
# ============================================================
# CELL 1 — IMPORTS + REPRODUCIBILITY + DEVICE
# ============================================================

import os
import gc
import sys
import json
import math
import time
import random
import zipfile
import importlib.util

from pathlib import Path
from collections import Counter

import numpy as np

# less GPU memory fragmentation (must be set before importing torch)
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import precision_recall_fscore_support


SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(SEED)

print("PyTorch:", torch.__version__)
print("Device :", DEVICE)

if DEVICE.type == "cuda":
    print("GPU    :", torch.cuda.get_device_name(0))

PyTorch: 2.11.0+cu128
Device : cuda
GPU    : Tesla T4


In [8]:
# ============================================================
# CELL 2 — CONFIGURATION
# Phase 3 (Rhetorical Roles): Architecture-1 DECODER (frozen) + IA3
#                             + sentence vectors + BiLSTM + CRF
# Trained on TRAIN only; dev only picks the best epoch.
# ============================================================

RUN_NAME = "p3_arch1_ia3"


# ------------------------------------------------------------
# Paths
# ------------------------------------------------------------

TOKENIZER_DIR = Path(
    "/kaggle/input/notebooks/ritikguptatab/updated-bpe-tokenizer/"
    "legal_tokenizer_v3/tokenizer/final_tokenizer"
)

# Phase-1 Architecture-1 checkpoint (best_model.pt of the decoder-only notebook).
# If wrong, CELL 9 searches /kaggle/input for it.
CHECKPOINT_PATH = Path(
    "/kaggle/input/notebooks/iitianritik/decoder-only-archi-1/legal_lm/best_model.pt"
)

# Phase-3 data: None = download from Hugging Face (Kaggle secret "HF"),
# or the folder that contains Train/ and Test/, e.g. Path("/kaggle/input/phase-3-rr/Phase_3_RR")
LOCAL_DATA_DIR = None
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"

OUTPUT_DIR = Path(f"/kaggle/working/{RUN_NAME}")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)


# ------------------------------------------------------------
# Sentences and windows
# (Architecture-1: causal decoder, RoPE, trained with 2048-token context)
# ------------------------------------------------------------

MAX_SENT_TOKENS = 256        # longer sentences are cut
WINDOW = 1024                # consecutive sentences are packed into windows of 1024 tokens
WINDOWS_PER_CHUNK = 8        # windows per training step (8 x 1024 tokens). If CUDA out-of-memory: 4

# ------------------------------------------------------------
# IA3 (Liu et al., 2022): learned rescaling vectors
#   keys   -> keys   * l_k        (k part of the fused qkv layer)
#   values -> values * l_v        (v part of the fused qkv layer)
#   FFN    -> down(hidden * l_ff) (hidden = SiLU(gate(x)) * up(x))
#   - Phase-1 weights stay FROZEN
#   - only l_k, l_v, l_ff are trained (they start at 1 -> no change)
# ------------------------------------------------------------

LR_ADAPTER = 3e-3            # IA3 has very few weights and needs a larger learning rate


# ------------------------------------------------------------
# Head: sentence vectors -> LayerNorm -> BiLSTM over the document -> CRF
#   sentence vector = [mean ; max] of its token vectors
#     mean = what the sentence is about
#     max  = whether it contains a strong cue (a case citation, "counsel submitted", ...)
# ------------------------------------------------------------

LSTM_HIDDEN = 256
HEAD_DROPOUT = 0.3
LR_HEAD = 2e-3

CLASS_BALANCING = True       # rare roles get a larger loss weight (macro-F1 counts every role equally)
CLASS_WEIGHT_MAX = 5.0       # cap, so a role with 3 sentences does not dominate
CE_WEIGHT = 0.5              # loss = CRF + CE_WEIGHT * class-weighted cross-entropy

# A role with fewer than MIN_LABEL_COUNT train sentences in a domain is never
# predicted in that domain (e.g. Dissent in IT: 0 sentences). Predicting a role
# that is not in the gold adds a 0 to the macro-F1 average.
MIN_LABEL_COUNT = 10


# ------------------------------------------------------------
# Training (TRAIN data only)
# ------------------------------------------------------------

EPOCHS = 15
PATIENCE = 5                 # stop if the dev score does not improve for 5 epochs
# Best epoch = highest dev macro-F1 over CL + IT TOGETHER ("all"): more sentences
# per role than each domain alone, so the choice is less noisy.
HEAD_ONLY_EPOCHS = 1         # epoch 1 trains only the head (the Phase-1 model is used as it is)
WARMUP_RATIO = 0.06
GRAD_CLIP = 1.0
TRAIN_EVAL_DOCS = 10         # a few train documents are scored every epoch (to see under-fitting)


# ------------------------------------------------------------
# Debug mode (quick run on a few documents)
# ------------------------------------------------------------

DEBUG_MODE = False
DEBUG_DOCS = 6               # per file

if DEBUG_MODE:
    EPOCHS = 2


# ------------------------------------------------------------
# Mixed precision (T4 / P100 -> float16 + GradScaler, A100 -> bfloat16)
# ------------------------------------------------------------

USE_AMP = DEVICE.type == "cuda"

if USE_AMP and torch.cuda.get_device_capability()[0] >= 8:
    AMP_DTYPE = torch.bfloat16
else:
    AMP_DTYPE = torch.float16

USE_SCALER = USE_AMP and AMP_DTYPE == torch.float16


def autocast():
    return torch.autocast(device_type="cuda", dtype=AMP_DTYPE, enabled=USE_AMP)


assert MAX_SENT_TOKENS <= WINDOW

print("Run       :", RUN_NAME)
print("Output    :", OUTPUT_DIR)
print("AMP dtype :", AMP_DTYPE if USE_AMP else "disabled")

Run       : p3_arch1_ia3
Output    : /kaggle/working/p3_arch1_ia3
AMP dtype : torch.float16


In [9]:
# ============================================================
# CELL 3 — LOAD THE PHASE-1 TOKENIZER
# ============================================================

spec = importlib.util.spec_from_file_location("legal_tokenizer_module", TOKENIZER_DIR / "tokenizer.py")
tok_module = importlib.util.module_from_spec(spec)
spec.loader.exec_module(tok_module)

tokenizer, tokenizer_config = tok_module.load_tokenizer(TOKENIZER_DIR)

PAD_ID = tokenizer.pad_id
UNK_ID = tokenizer.unk_id
VOCAB_SIZE = tokenizer.get_vocab_size()

print("Vocabulary size:", VOCAB_SIZE, "| PAD / UNK:", PAD_ID, UNK_ID)


token_cache = {}      # sentence -> token ids (legal documents repeat many sentences)


def encode_sentence(sentence):
    """
    Token ids of one sentence (NOT cut yet; CELL 6 cuts to MAX_SENT_TOKENS).
    A space is added in front: inside a document every sentence follows a
    space, and the tokenizer keeps the space in front of the first word.
    """

    if sentence not in token_cache:

        ids = tokenizer.encode(" " + sentence.strip())

        token_cache[sentence] = ids if ids else [UNK_ID]   # empty sentence -> 1 token, so it still gets a vector

    return token_cache[sentence]


demo = "The petitioner filed the present appeal."
print("\nDemo:", [tokenizer.id_to_token[i] for i in encode_sentence(demo)])

Vocabulary size: 32004 | PAD / UNK: 0 1

Demo: [' ', 'The', ' ', 'petitioner', ' ', 'filed', ' ', 'the', ' ', 'present', ' ', 'appeal', '.']


In [10]:
# ============================================================
# CELL 4 — PHASE-3 DATA FILES  (download needs the Kaggle secret "HF")
# ============================================================

PHASE3_FILES = {
    "CL_train": "Phase_3_RR/Train/CL_train.json",
    "CL_dev":   "Phase_3_RR/Train/CL_dev.json",
    "IT_train": "Phase_3_RR/Train/IT_train.json",
    "IT_dev":   "Phase_3_RR/Train/IT_dev.json",
    "labels":   "Phase_3_RR/Train/label_vocab.json",
    "CL_test":  "Phase_3_RR/Test/CL_test.json",
    "IT_test":  "Phase_3_RR/Test/IT_test.json",
}

DATA_FILES = {}

if LOCAL_DATA_DIR is not None:

    for name, filename in PHASE3_FILES.items():
        DATA_FILES[name] = Path(LOCAL_DATA_DIR) / filename.replace("Phase_3_RR/", "")

else:

    from huggingface_hub import hf_hub_download
    from kaggle_secrets import UserSecretsClient

    HF_TOKEN = UserSecretsClient().get_secret("HF")

    for name, filename in PHASE3_FILES.items():

        DATA_FILES[name] = hf_hub_download(
            repo_id=HF_DATASET_REPO,
            filename=filename,
            repo_type="dataset",
            token=HF_TOKEN,
        )

for name, path in DATA_FILES.items():
    print(f"{name:9s} -> {path}")

CL_train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/CL_train.json
CL_dev    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/CL_dev.json
IT_train  -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/IT_train.json
IT_dev    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/IT_dev.json
labels    -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Train/label_vocab.json
CL_test   -> /root/.cache/huggingface/hub/datasets--Exploration-Lab--CS-781-Capstone/snapshots/89edba6a44d6bb9593fd13b6895b5cd3b06eb82e/Phase_3_RR/Test/CL_test.json
I

In [11]:
# ============================================================
# CELL 5 — LOAD DOCUMENTS + LABELS + STATISTICS
#
# Every document: {"id", "domain", "sentences": [...], "labels": [...] or None}
# ============================================================

def read_json(path):
    """A JSON file, or a JSON-lines file (one document per line)."""

    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    try:
        return json.loads(content)
    except json.JSONDecodeError:
        return [json.loads(line) for line in content.splitlines() if line.strip()]


def load_label_vocab(path):

    raw = read_json(path)

    if isinstance(raw, dict) and isinstance(raw.get("labels"), list):
        raw = raw["labels"]

    if isinstance(raw, list):
        return [str(label) for label in raw]

    # dict: {label: id} or {id: label}
    if all(isinstance(v, int) for v in raw.values()):
        return [label for label, _ in sorted(raw.items(), key=lambda item: item[1])]

    return [str(raw[key]) for key in sorted(raw, key=int)]


def load_documents(path, domain):

    raw = read_json(path)

    # accepted layouts: [doc, ...]  |  {"data": [doc, ...]}  |  {doc_id: doc}
    if isinstance(raw, dict):

        list_key = next((k for k in ("data", "documents", "docs") if isinstance(raw.get(k), list)), None)

        if list_key is not None:
            raw = raw[list_key]
        else:
            raw = [
                dict(value, id=key) if isinstance(value, dict) else {"id": key, "text": value}
                for key, value in raw.items()
            ]

    docs = []

    for i, item in enumerate(raw):

        sentences = item.get("text", item.get("sentences"))
        labels = item.get("labels", item.get("label"))

        doc = {
            "id": str(item.get("id", f"{domain}_{i}")),
            "domain": domain,
            "sentences": [str(s) for s in sentences],
            "labels": None if labels is None else [str(label) for label in labels],
        }

        if doc["labels"] is not None:
            assert len(doc["labels"]) == len(doc["sentences"]), f"{doc['id']}: labels and sentences differ in length"

        docs.append(doc)

    if DEBUG_MODE:
        docs = docs[:DEBUG_DOCS]

    return docs


LABELS = load_label_vocab(DATA_FILES["labels"])
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
NUM_LABELS = len(LABELS)

print(f"{NUM_LABELS} labels:", LABELS)

train_docs = load_documents(DATA_FILES["CL_train"], "CL") + load_documents(DATA_FILES["IT_train"], "IT")
dev_docs = load_documents(DATA_FILES["CL_dev"], "CL") + load_documents(DATA_FILES["IT_dev"], "IT")
test_docs = load_documents(DATA_FILES["CL_test"], "CL") + load_documents(DATA_FILES["IT_test"], "IT")


# ------------------------------------------------------------
# Label ids. A label that is not in label_vocab.json is reported
# and mapped to "None".
# ------------------------------------------------------------

unknown = Counter()

for doc in train_docs + dev_docs:

    for k, label in enumerate(doc["labels"]):
        if label not in LABEL2ID:
            unknown[label] += 1
            doc["labels"][k] = "None"

    doc["label_ids"] = [LABEL2ID[label] for label in doc["labels"]]

if unknown:
    print("\nWARNING: labels not in label_vocab.json (mapped to 'None'):", dict(unknown))


# ------------------------------------------------------------
# Statistics
# ------------------------------------------------------------

print(f"\n{'split':6s} {'domain':6s} {'docs':>5s} {'sentences':>10s} {'per doc (mean / max)':>22s}")

for split, docs in [("train", train_docs), ("dev", dev_docs), ("test", test_docs)]:
    for domain in ["CL", "IT"]:

        lengths = [len(d["sentences"]) for d in docs if d["domain"] == domain]

        if lengths:
            print(f"{split:6s} {domain:6s} {len(lengths):5d} {sum(lengths):10d} "
                  f"{np.mean(lengths):14.1f} / {max(lengths):5d}")

print(f"\nLabel distribution (train | dev):")
print(f"{'label':24s} {'CL train':>9s} {'IT train':>9s} {'CL dev':>7s} {'IT dev':>7s}")

counts = {
    (split, domain): Counter(label for d in docs if d["domain"] == domain for label in d["labels"])
    for split, docs in [("train", train_docs), ("dev", dev_docs)]
    for domain in ["CL", "IT"]
}

for label in LABELS:
    print(f"{label:24s} {counts['train', 'CL'][label]:9d} {counts['train', 'IT'][label]:9d} "
          f"{counts['dev', 'CL'][label]:7d} {counts['dev', 'IT'][label]:7d}")


# ------------------------------------------------------------
# Class weights for the loss: (N / (C * n_label)) ** 0.5, at most CLASS_WEIGHT_MAX.
# The square root keeps the weights moderate.
# ------------------------------------------------------------

train_counts = Counter(label for d in train_docs for label in d["labels"])
n_train = sum(train_counts.values())

if CLASS_BALANCING:
    CLASS_WEIGHTS = torch.tensor(
        [min(CLASS_WEIGHT_MAX, (n_train / (NUM_LABELS * max(train_counts[label], 1))) ** 0.5) for label in LABELS]
    )
else:
    CLASS_WEIGHTS = torch.ones(NUM_LABELS)

print("\nClass weights:", {label: round(w, 2) for label, w in zip(LABELS, CLASS_WEIGHTS.tolist())})


# ------------------------------------------------------------
# Roles allowed per domain (used at prediction time, CELL 10)
# ------------------------------------------------------------

LABEL_ALLOWED = {
    domain: np.array([counts["train", domain][label] >= MIN_LABEL_COUNT for label in LABELS])
    for domain in ["CL", "IT"]
}

for domain in ["CL", "IT"]:
    never = [label for label, ok in zip(LABELS, LABEL_ALLOWED[domain]) if not ok]
    print(f"Never predicted in {domain} (< {MIN_LABEL_COUNT} train sentences):", never)
print("\nSample:", train_docs[0]["id"], "|", train_docs[0]["labels"][0], "|", train_docs[0]["sentences"][0][:120])


split  domain  docs  sentences   per doc (mean / max)
train  CL        40      10593          264.8 /  1816
train  IT        40       6252          156.3 /   442
dev    CL         5       1396          279.2 /   848
dev    IT         5        746          149.2 /   382
test   CL         5       1339          267.8 /   942
test   IT         5        858          171.6 /   420

Label distribution (train | dev):
label                     CL train  IT train  CL dev  IT dev
Fact                          2979      1119     249     195
Issue                           17        90       3      12
ArgumentPetitioner             976       637     344      56
ArgumentRespondent             996       352     236      36
Statute                        367       732      58      62
Dissent                        284         0       0       0
PrecedentReliedUpon           1305       640     142     112
PrecedentNotReliedUpon         209       360       0      50
PrecedentOverruled               0   

In [12]:
# ============================================================
# CELL 6 — TOKENIZE + WINDOWS + CHUNKS
#
# window = token ids that go through the model together,
#          {"ids": [...], "sentences": [(sentence_index, start, end), ...]}
#   consecutive sentences are packed into one window (up to WINDOW
#   tokens). The model is causal, so every sentence sees the sentences
#   BEFORE it in its window; the BiLSTM of the head adds the ones after.
#
# chunk = the windows of ONE training step (consecutive sentences
#         of one document).
# ============================================================

def pack_windows(doc):

    windows = []

    for i, sentence in enumerate(doc["sentences"]):

        ids = encode_sentence(sentence)[:MAX_SENT_TOKENS]

        if not windows or len(windows[-1]["ids"]) + len(ids) > WINDOW:
            windows.append({"ids": [], "sentences": []})

        window = windows[-1]

        window["sentences"].append((i, len(window["ids"]), len(window["ids"]) + len(ids)))
        window["ids"] = window["ids"] + ids

    return windows


def make_chunks(docs):

    chunks = []

    for doc in docs:
        for k in range(0, len(doc["windows"]), WINDOWS_PER_CHUNK):
            chunks.append({"doc": doc, "windows": doc["windows"][k:k + WINDOWS_PER_CHUNK]})

    return chunks


t0 = time.time()

for doc in train_docs + dev_docs + test_docs:

    doc["windows"] = pack_windows(doc)

    # every sentence must be in exactly one window, in order
    order = [i for window in doc["windows"] for i, _, _ in window["sentences"]]
    assert order == list(range(len(doc["sentences"]))), doc["id"]

print(f"Tokenized in {time.time() - t0:.1f}s")

lengths = np.array([len(encode_sentence(s)) for d in train_docs for s in d["sentences"]])

print(f"Sentences longer than {MAX_SENT_TOKENS} tokens (cut): {(lengths > MAX_SENT_TOKENS).mean():.2%}")
print(f"Train windows: {sum(len(d['windows']) for d in train_docs)} | "
      f"train chunks (steps per epoch): {len(make_chunks(train_docs))}")

Tokenized in 6.0s
Sentences longer than 256 tokens (cut): 1.92%
Train windows: 1435 | train chunks (steps per epoch): 212


In [13]:
# ============================================================
# CELL 7 — SCORES + VITERBI
# ============================================================

def macro_scores(gold, pred):
    """Macro precision / recall / F1 over the labels that occur in gold or prediction."""

    labels = sorted(set(gold) | set(pred))

    p, r, f, _ = precision_recall_fscore_support(gold, pred, labels=labels, average="macro", zero_division=0)

    return {"precision": float(p), "recall": float(r), "f1": float(f)}


def flat_labels(docs, predictions, domain=None):
    """Gold and predicted labels of all sentences, in order (one domain or all)."""

    gold, pred = [], []

    for doc in docs:
        if domain is None or doc["domain"] == domain:
            gold.extend(doc["labels"])
            pred.extend(predictions[doc["id"]])

    return gold, pred


def domain_scores(docs, predictions):
    """
    Macro scores for CL, IT and all sentences together.
    "score" = mean of the CL and IT macro-F1 (both domains count equally);
    it is used to pick the best epoch.
    """

    scores = {}

    for domain in ["CL", "IT"]:

        gold, pred = flat_labels(docs, predictions, domain)

        if gold:
            scores[domain] = macro_scores(gold, pred)

    scores["all"] = macro_scores(*flat_labels(docs, predictions))
    scores["score"] = float(np.mean([scores[d]["f1"] for d in ["CL", "IT"] if d in scores]))

    return scores


def viterbi(emissions, transitions, start_scores, end_scores):
    """emissions [n_sentences, NUM_LABELS] -> best label id per sentence (numpy)."""

    n = len(emissions)

    score = start_scores + emissions[0]
    backpointer = np.zeros((n, NUM_LABELS), dtype=np.int64)

    for i in range(1, n):
        candidates = score[:, None] + transitions          # [previous, current]
        backpointer[i] = candidates.argmax(axis=0)
        score = candidates.max(axis=0) + emissions[i]

    path = [int((score + end_scores).argmax())]

    for i in range(n - 1, 0, -1):
        path.append(int(backpointer[i, path[-1]]))

    return np.array(path[::-1], dtype=np.int64)


_g = ["Fact", "Fact", "Issue", "Statute"]
_p = ["Fact", "Issue", "Issue", "Statute"]
print("macro_scores check:", macro_scores(_g, _p))

macro_scores check: {'precision': 0.8333333333333334, 'recall': 0.8333333333333334, 'f1': 0.7777777777777777}


In [14]:
# ============================================================
# CELL 8 — MODEL
#
#   sentences -> Architecture-1 DECODER (Phase-1 weights FROZEN, + IA3)
#             -> [mean ; max] of the token vectors of each sentence = sentence vector
#             -> LayerNorm, + relative position in the document + domain flag
#             -> BiLSTM over the sentences of the document -> CRF
#
# Layer names match the Phase-1 notebook, so the weights load directly.
# ============================================================

# ============================================================
# Architecture-1 DECODER
# ============================================================

class RMSNorm(nn.Module):

    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        x_float = x.float()
        rms = torch.rsqrt(x_float.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return (x_float * rms * self.weight).to(x.dtype)


def build_rope_cache(block_size, head_dim, theta):

    inv_freq = 1.0 / (theta ** (torch.arange(0, head_dim, 2).float() / head_dim))
    angles = torch.outer(torch.arange(block_size).float(), inv_freq)

    return angles.cos(), angles.sin()


def apply_rope(x, cos, sin):

    x1, x2 = x.float().chunk(2, dim=-1)

    return torch.cat([x1 * cos - x2 * sin, x1 * sin + x2 * cos], dim=-1).to(x.dtype)


class CausalSelfAttention(nn.Module):

    def __init__(self, config):
        super().__init__()

        d_model = config["d_model"]

        self.n_heads = config["n_heads"]
        self.head_dim = d_model // self.n_heads

        self.qkv = nn.Linear(d_model, 3 * d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)

        self.q_norm = RMSNorm(self.head_dim)
        self.k_norm = RMSNorm(self.head_dim)

    def forward(self, x, cos, sin):

        B, T, C = x.shape

        q, k, v = self.qkv(x).split(C, dim=-1)

        q = q.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        k = k.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)
        v = v.view(B, T, self.n_heads, self.head_dim).transpose(1, 2)

        q = apply_rope(self.q_norm(q), cos, sin)
        k = apply_rope(self.k_norm(k), cos, sin)

        # causal, as in pretraining. Padding is at the END, so real tokens never see it.
        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)

        return self.out_proj(y.transpose(1, 2).contiguous().view(B, T, C))


class SwiGLU(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.gate = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.up = nn.Linear(config["d_model"], config["d_ff"], bias=False)
        self.down = nn.Linear(config["d_ff"], config["d_model"], bias=False)

    def forward(self, x):
        return self.down(F.silu(self.gate(x)) * self.up(x))


class TransformerBlock(nn.Module):

    def __init__(self, config):
        super().__init__()

        self.norm1 = RMSNorm(config["d_model"])
        self.attn = CausalSelfAttention(config)
        self.norm2 = RMSNorm(config["d_model"])
        self.mlp = SwiGLU(config)

    def forward(self, x, cos, sin):

        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.mlp(self.norm2(x))

        return x


class DecoderBackbone(nn.Module):
    """Architecture-1 without the LM head: tokens -> hidden vectors."""

    def __init__(self, config):
        super().__init__()

        self.token_embedding = nn.Embedding(config["vocab_size"], config["d_model"])
        self.blocks = nn.ModuleList([TransformerBlock(config) for _ in range(config["n_layers"])])
        self.norm_f = RMSNorm(config["d_model"])

        cos, sin = build_rope_cache(config["block_size"], config["d_model"] // config["n_heads"], config["rope_theta"])
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, input_ids, mask=None):
        """`mask` is not needed: attention is causal and padding is at the end."""

        T = input_ids.size(1)

        cos, sin = self.rope_cos[:T], self.rope_sin[:T]

        x = self.token_embedding(input_ids)

        for block in self.blocks:
            x = block(x, cos, sin)

        return self.norm_f(x)


# ============================================================
# IA3 (learned rescaling vectors, start at 1 = no change)
#   IA3QKV  : keys * l_k and values * l_v inside the fused qkv layer
#   IA3Input: down(hidden * l_ff), the FFN hidden
# ============================================================

class IA3QKV(nn.Module):

    def __init__(self, base):
        super().__init__()

        self.base = base                                      # frozen fused q/k/v layer
        d_model = base.out_features // 3

        self.ia3_k = nn.Parameter(torch.ones(d_model))
        self.ia3_v = nn.Parameter(torch.ones(d_model))

    def forward(self, x):

        q, k, v = self.base(x).chunk(3, dim=-1)

        return torch.cat([q, k * self.ia3_k.to(k.dtype), v * self.ia3_v.to(v.dtype)], dim=-1)


class IA3Output(nn.Module):
    """layer(x) * l"""

    def __init__(self, base):
        super().__init__()

        self.base = base                                      # frozen Phase-1 layer
        self.ia3_scale = nn.Parameter(torch.ones(base.out_features))

    def forward(self, x):

        h = self.base(x)

        return h * self.ia3_scale.to(h.dtype)


class IA3Input(nn.Module):
    """layer(x * l)"""

    def __init__(self, base):
        super().__init__()

        self.base = base                                      # frozen Phase-1 layer
        self.ia3_scale = nn.Parameter(torch.ones(base.in_features))

    def forward(self, x):

        return self.base(x * self.ia3_scale.to(x.dtype))


def add_adapters(decoder):
    """Freeze every Phase-1 weight, then add the IA3 vectors to every block."""

    for p in decoder.parameters():
        p.requires_grad = False

    for block in decoder.blocks:

        block.attn.qkv = IA3QKV(block.attn.qkv)               # l_k, l_v
        block.mlp.down = IA3Input(block.mlp.down)             # l_ff on SiLU(gate(x)) * up(x)

# ============================================================
# CRF + sentence-role model
# ============================================================

class CRF(nn.Module):
    """Linear-chain CRF over the sentences of a document (learns which role follows which)."""

    def __init__(self, num_labels):
        super().__init__()

        self.transitions = nn.Parameter(torch.zeros(num_labels, num_labels))      # [previous, current]
        self.start = nn.Parameter(torch.zeros(num_labels))
        self.end = nn.Parameter(torch.zeros(num_labels))

    def loss(self, emissions, labels):
        """Negative log-likelihood per sentence. emissions [S, C] (float32), labels [S]."""

        S = emissions.size(0)

        gold = (
            self.start[labels[0]]
            + emissions[torch.arange(S, device=emissions.device), labels].sum()
            + self.transitions[labels[:-1], labels[1:]].sum()
            + self.end[labels[-1]]
        )

        alpha = self.start + emissions[0]

        for i in range(1, S):
            alpha = torch.logsumexp(alpha.unsqueeze(1) + self.transitions, dim=0) + emissions[i]

        log_z = torch.logsumexp(alpha + self.end, dim=0)

        return (log_z - gold) / S


class SentenceRoleModel(nn.Module):
    """Architecture-1 decoder (+ IA3) + head."""

    def __init__(self, backbone, d_model):
        super().__init__()

        self.backbone = backbone
        self.dropout = nn.Dropout(HEAD_DROPOUT)

        # LayerNorm: sentence vectors of any size -> mean 0, std 1 (keeps the BiLSTM from saturating)
        self.vector_norm = nn.LayerNorm(2 * d_model)          # [mean ; max] -> 2 * d_model

        self.lstm = nn.LSTM(2 * d_model + 2, LSTM_HIDDEN, batch_first=True, bidirectional=True)
        self.classifier = nn.Linear(2 * LSTM_HIDDEN, NUM_LABELS)
        self.crf = CRF(NUM_LABELS)

        self.register_buffer("class_weights", CLASS_WEIGHTS.clone())

    def head_parameters(self):
        return [p for name, p in self.named_parameters() if not name.startswith("backbone.")]

    def sentence_vectors(self, windows):
        """windows -> (vectors [S, 2D] float32, sentence_index [S]).
        Sentence vector = [mean ; max] of its token vectors."""

        max_len = max(len(w["ids"]) for w in windows)
        ids = torch.full((len(windows), max_len), PAD_ID, dtype=torch.long)
        mask = torch.zeros((len(windows), max_len), dtype=torch.bool)

        for row, window in enumerate(windows):
            ids[row, :len(window["ids"])] = torch.tensor(window["ids"])
            mask[row, :len(window["ids"])] = True

        with autocast():
            hidden = self.backbone(ids.to(DEVICE), mask.to(DEVICE))           # [W, T, D]

        vectors = []
        sentence_index = []

        for row, window in enumerate(windows):
            for i, start, end in window["sentences"]:
                tokens = hidden[row, start:end].float()                      # [n_tokens, D]
                vectors.append(torch.cat([tokens.mean(dim=0), tokens.max(dim=0).values]))
                sentence_index.append(i)

        return torch.stack(vectors), sentence_index

    def head(self, vectors, doc, sentence_index):
        """Sentence vectors [S, 2D] -> label scores [S, NUM_LABELS] (float32)."""

        n = len(doc["sentences"])

        position = torch.tensor([[i / max(n - 1, 1)] for i in sentence_index], device=DEVICE)
        domain = torch.full_like(position, 1.0 if doc["domain"] == "IT" else 0.0)

        x = self.dropout(torch.cat([self.vector_norm(vectors), position, domain], dim=-1))

        out, _ = self.lstm(x.unsqueeze(0))              # one sequence = the sentences in document order

        return self.classifier(self.dropout(out.squeeze(0)))

    def chunk_emissions(self, chunk):
        """Label scores of the sentences of one training chunk."""

        vectors, sentence_index = self.sentence_vectors(chunk["windows"])

        return self.head(vectors, chunk["doc"], sentence_index), sentence_index

# ------------------------------------------------------------
# Check: IA3 really rescales
# ------------------------------------------------------------

_x = torch.randn(3, 8)

_qkv = IA3QKV(nn.Linear(8, 12, bias=False))
nn.init.normal_(_qkv.ia3_k)
nn.init.normal_(_qkv.ia3_v)
_q, _k, _v = _qkv.base(_x).chunk(3, dim=-1)
assert torch.allclose(_qkv(_x), torch.cat([_q, _k * _qkv.ia3_k, _v * _qkv.ia3_v], dim=-1), atol=1e-6)

_inp = IA3Input(nn.Linear(8, 6, bias=False))
nn.init.normal_(_inp.ia3_scale)
assert torch.allclose(_inp(_x), _inp.base(_x * _inp.ia3_scale), atol=1e-6)

print("✅ IA3: k * l_k, v * l_v, down(hidden * l_ff)")

✅ IA3: k * l_k, v * l_v, down(hidden * l_ff)


In [15]:
# ============================================================
# CELL 9 — BUILD THE MODEL FROM THE PHASE-1 ARCHITECTURE-1 CHECKPOINT
# ============================================================

def is_architecture1(path):

    try:
        ckpt = torch.load(path, map_location="cpu", weights_only=False)
        return isinstance(ckpt, dict) and "rope_theta" in ckpt.get("model_config", {})
    except Exception:
        return False


if not CHECKPOINT_PATH.exists():

    found = [p for p in sorted(Path("/kaggle/input").rglob("best_model.pt")) if is_architecture1(p)]
    assert found, f"{CHECKPOINT_PATH} not found and no Architecture-1 checkpoint under /kaggle/input"

    CHECKPOINT_PATH = found[0]
    print("Checkpoint found at:", CHECKPOINT_PATH)


phase1 = torch.load(CHECKPOINT_PATH, map_location="cpu", weights_only=False)
MODEL_CONFIG = phase1["model_config"]

# decoder weights without the LM head (tied to the embedding, not needed here)
MODEL_WEIGHTS = {
    key.replace("_orig_mod.", ""): value
    for key, value in phase1["model_state_dict"].items()
    if not key.replace("_orig_mod.", "").startswith("lm_head.")
}

del phase1

print("Phase-1 config:", MODEL_CONFIG)

assert MODEL_CONFIG["vocab_size"] == VOCAB_SIZE, "checkpoint was trained with a different tokenizer"
assert WINDOW <= MODEL_CONFIG["block_size"], f"WINDOW must be <= {MODEL_CONFIG['block_size']}"


def build_model():
    """Frozen Phase-1 decoder + IA3 + new head."""

    backbone = DecoderBackbone(MODEL_CONFIG)
    backbone.load_state_dict(MODEL_WEIGHTS, strict=True)          # every name and shape must match

    # the adapters start as "no change": the output must stay the same
    test_ids = torch.randint(4, VOCAB_SIZE, (1, 24))
    test_mask = torch.ones(1, 24, dtype=torch.bool)

    backbone.eval()
    with torch.no_grad():
        before = backbone(test_ids, test_mask)
        add_adapters(backbone)
        after = backbone(test_ids, test_mask)
    backbone.train()

    assert torch.allclose(before, after, atol=1e-4), "IA3 changed the model at the start"

    return SentenceRoleModel(backbone, MODEL_CONFIG["d_model"]).to(DEVICE)


# free GPU memory if this cell is re-run
for name in ["model", "optimizer", "scaler"]:
    globals().pop(name, None)

sys.last_traceback = sys.last_value = sys.last_type = None
gc.collect()

if DEVICE.type == "cuda":
    torch.cuda.empty_cache()

model = build_model()

n_total = sum(p.numel() for p in model.parameters())
n_backbone = sum(p.numel() for p in model.backbone.parameters())
n_adapter = sum(p.numel() for p in model.backbone.parameters() if p.requires_grad)
n_head = sum(p.numel() for p in model.head_parameters())
n_trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)

assert n_trainable == n_adapter + n_head

print(f"\nPhase-1 model (frozen)   : {n_backbone - n_adapter:,}")
print(f"IA3 (trainable)          : {n_adapter:,}  ({n_adapter / (n_backbone - n_adapter):.3%} of the Phase-1 model)")
print(f"Head (trainable)         : {n_head:,}")
print(f"Trainable total          : {n_trainable:,}  ({n_trainable / n_total:.2%} of {n_total:,})")


# ------------------------------------------------------------
# Size of the sentence vectors BEFORE the LayerNorm of the head
# ------------------------------------------------------------

model.eval()
with torch.no_grad():
    _vectors, _ = model.sentence_vectors(make_chunks(train_docs[:1])[0]["windows"])
model.train()

_cos = F.cosine_similarity(_vectors[:, None], _vectors[None], dim=-1).mean()

print(f"\nSentence vectors: mean |value| {_vectors.abs().mean():.2f} | max |value| {_vectors.abs().max():.1f} | "
      f"mean cosine between sentences {_cos:.3f}")

Phase-1 config: {'vocab_size': 32004, 'block_size': 2048, 'd_model': 768, 'n_layers': 12, 'n_heads': 12, 'd_ff': 2048, 'rope_theta': 10000.0}

Phase-1 model (frozen)   : 109,534,464
IA3 (trainable)          : 43,008  (0.039% of the Phase-1 model)
Head (trainable)         : 3,688,144
Trainable total          : 3,731,152  (3.29% of 113,265,616)

Sentence vectors: mean |value| 1.85 | max |value| 15.8 | mean cosine between sentences 0.943


In [16]:
# ============================================================
# CELL 10 — PREDICTION + EVALUATION
#
# A whole document goes through the model (windows in batches),
# then the head sees all its sentences at once and the CRF Viterbi
# picks the best role sequence (use_crf=False: every sentence simply
# gets its own best role, to check what the CRF adds). Roles not allowed
# in the document's domain (CELL 5) are never picked. One role per sentence.
# ============================================================

EVAL_WINDOWS = 8          # windows per forward pass


@torch.no_grad()
def document_emissions(model, doc):
    """Label scores of every sentence of one document: numpy [n_sentences, NUM_LABELS].
    The model must be in eval mode."""

    vectors, sentence_index = [], []

    for k in range(0, len(doc["windows"]), EVAL_WINDOWS):

        v, idx = model.sentence_vectors(doc["windows"][k:k + EVAL_WINDOWS])

        vectors.append(v)
        sentence_index.extend(idx)

    return model.head(torch.cat(vectors), doc, sentence_index).float().cpu().numpy()


@torch.no_grad()
def predict_labels(model, docs, use_crf=True):
    """{doc_id: [label, label, ...]}  one label per sentence, in order."""

    model.eval()

    transitions = model.crf.transitions.float().cpu().numpy()
    start = model.crf.start.float().cpu().numpy()
    end = model.crf.end.float().cpu().numpy()

    predictions = {}

    for doc in docs:

        if not doc["sentences"]:
            predictions[doc["id"]] = []
            continue

        emissions = document_emissions(model, doc)
        emissions[:, ~LABEL_ALLOWED[doc["domain"]]] = -1e4

        if use_crf:
            label_ids = viterbi(emissions, transitions, start, end)
        else:
            label_ids = emissions.argmax(axis=1)

        predictions[doc["id"]] = [LABELS[i] for i in label_ids]

    model.train()

    return predictions


def evaluate(model, docs, use_crf=True):

    predictions = predict_labels(model, docs, use_crf)

    return domain_scores(docs, predictions), predictions

In [17]:
# ============================================================
# CELL 11 — LOSS + OPTIMIZER + TRAINING FUNCTION
#   loss = CRF + CE_WEIGHT * class-weighted cross-entropy
#   group 0 = IA3 weights, group 1 = head
# ============================================================

def chunk_loss(model, chunk):

    doc = chunk["doc"]

    emissions, sentence_index = model.chunk_emissions(chunk)
    labels = torch.tensor([doc["label_ids"][i] for i in sentence_index], device=DEVICE)

    return model.crf.loss(emissions, labels) + CE_WEIGHT * F.cross_entropy(emissions, labels, weight=model.class_weights)


def make_optimizer(model, total_steps):

    optimizer = torch.optim.AdamW(
        [
            {"params": [p for p in model.backbone.parameters() if p.requires_grad], "lr": LR_ADAPTER, "weight_decay": 0.0},
            {"params": model.head_parameters(), "lr": LR_HEAD, "weight_decay": 0.0},
        ],
        betas=(0.9, 0.999),
    )

    base_lrs = [group["lr"] for group in optimizer.param_groups]
    warmup_steps = max(1, int(WARMUP_RATIO * total_steps))

    def set_lr(step):
        """Linear warmup, then linear decay to 0."""

        if step < warmup_steps:
            factor = (step + 1) / warmup_steps
        else:
            factor = max(0.0, (total_steps - step) / max(1, total_steps - warmup_steps))

        for group, base_lr in zip(optimizer.param_groups, base_lrs):
            group["lr"] = base_lr * factor

    return optimizer, set_lr


def trainable_state(model):
    """Only the trained weights (IA3 + head): small, quick to copy."""

    return {name: p.detach().cpu().clone() for name, p in model.named_parameters() if p.requires_grad}


def train_model(model, epochs):
    """
    Trains on TRAIN documents only. After every epoch the model is scored on dev;
    the best epoch (dev macro-F1, CL + IT together) is kept, and training stops
    after PATIENCE epochs without improvement.
    Returns (history, best_epoch).
    """

    chunks = make_chunks(train_docs)

    optimizer, set_lr = make_optimizer(model, epochs * len(chunks))
    scaler = torch.amp.GradScaler("cuda", enabled=USE_SCALER)

    train_check_docs = random.Random(SEED).sample(train_docs, min(TRAIN_EVAL_DOCS, len(train_docs)))

    history = []
    best_score, best_epoch, best_state = -1.0, 0, None
    step = 0
    t_start = time.time()

    model.train()

    for epoch in range(1, epochs + 1):

        random.shuffle(chunks)
        total_loss = 0.0
        skipped = 0                   # steps skipped by the float16 GradScaler (overflow)
        head_only = epoch <= HEAD_ONLY_EPOCHS

        for chunk in chunks:

            set_lr(step)

            if head_only:
                optimizer.param_groups[0]["lr"] = 0.0          # group 0 = adapter: not trained yet

            loss = chunk_loss(model, chunk)

            optimizer.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.unscale_(optimizer)

            # clip adapter and head separately
            torch.nn.utils.clip_grad_norm_(optimizer.param_groups[0]["params"], GRAD_CLIP)
            torch.nn.utils.clip_grad_norm_(optimizer.param_groups[1]["params"], GRAD_CLIP)

            scale_before = scaler.get_scale()
            scaler.step(optimizer)
            scaler.update()
            skipped += scaler.get_scale() < scale_before

            step += 1
            total_loss += loss.item()

        train_scores, _ = evaluate(model, train_check_docs)
        dev_scores, _ = evaluate(model, dev_docs)

        log = {"epoch": epoch, "loss": total_loss / len(chunks), "train_f1": train_scores["score"],
               "dev": dev_scores, "head_only": head_only, "skipped_steps": int(skipped)}

        line = (f"Epoch {epoch:2d}/{epochs}{' (head only)' if head_only else ''} | loss {log['loss']:.4f} | "
                f"train F1 {log['train_f1']:.4f} | dev F1 CL {dev_scores.get('CL', {}).get('f1', 0):.4f} "
                f"IT {dev_scores.get('IT', {}).get('f1', 0):.4f} all {dev_scores['all']['f1']:.4f}")

        stop = False

        if dev_scores["all"]["f1"] > best_score:                 # CL + IT together
            best_score, best_epoch, best_state = dev_scores["all"]["f1"], epoch, trainable_state(model)
            line += "  <- best"
        elif epoch - best_epoch >= PATIENCE:
            stop = True

        if skipped:
            line += f" | skipped {skipped}"
        line += f" | {(time.time() - t_start) / 60:.1f} min"

        print(line, flush=True)
        history.append(log)

        if stop:
            print(f"Early stop: no improvement for {PATIENCE} epochs.")
            break

    model.load_state_dict(best_state, strict=False)          # back to the best epoch

    return history, best_epoch

In [18]:
# ============================================================
# CELL 12 — TRAIN (TRAIN data only, best epoch picked on dev)
# ============================================================

print(f"{RUN_NAME} | train {len(train_docs)} docs -> dev {len(dev_docs)} docs")

if DEVICE.type == "cuda":
    torch.cuda.reset_peak_memory_stats()

t0 = time.time()
history, best_epoch = train_model(model, EPOCHS)
train_minutes = (time.time() - t0) / 60

peak_gpu_gb = torch.cuda.max_memory_allocated() / 1e9 if DEVICE.type == "cuda" else 0.0

torch.save(trainable_state(model), OUTPUT_DIR / "best_trainable.pt")

print(f"\n✅ Trained in {train_minutes:.1f} min | best epoch {best_epoch} | peak GPU {peak_gpu_gb:.2f} GB")

p3_arch1_ia3 | train 80 docs -> dev 10 docs
Epoch  1/15 (head only) | loss 3.3098 | train F1 0.0970 | dev F1 CL 0.0926 IT 0.1010 all 0.0906  <- best | 1.9 min
Epoch  2/15 | loss 3.0053 | train F1 0.1446 | dev F1 CL 0.1420 IT 0.1436 all 0.1365  <- best | 4.0 min
Epoch  3/15 | loss 2.5518 | train F1 0.1807 | dev F1 CL 0.1439 IT 0.1828 all 0.1589  <- best | 6.1 min
Epoch  4/15 | loss 2.1457 | train F1 0.2687 | dev F1 CL 0.2674 IT 0.2348 all 0.2248  <- best | 8.1 min
Epoch  5/15 | loss 1.8622 | train F1 0.3651 | dev F1 CL 0.2572 IT 0.3142 all 0.2691  <- best | 10.2 min
Epoch  6/15 | loss 1.6294 | train F1 0.3621 | dev F1 CL 0.3987 IT 0.3337 all 0.3475  <- best | 12.2 min
Epoch  7/15 | loss 1.4980 | train F1 0.4498 | dev F1 CL 0.3710 IT 0.3340 all 0.3627  <- best | 14.3 min
Epoch  8/15 | loss 1.3904 | train F1 0.4373 | dev F1 CL 0.3660 IT 0.2872 all 0.3158 | 16.4 min
Epoch  9/15 | loss 1.2907 | train F1 0.4711 | dev F1 CL 0.3286 IT 0.4340 all 0.4188  <- best | 18.4 min
Epoch 10/15 | loss 1.

In [19]:
# ============================================================
# CELL 13 — DEV REPORT: decoding check, per domain, per label, confusions + save results
# ============================================================

t0 = time.time()
crf_scores, crf_predictions = evaluate(model, dev_docs, use_crf=True)
predict_seconds = time.time() - t0

plain_scores, plain_predictions = evaluate(model, dev_docs, use_crf=False)


# ------------------------------------------------------------
# Decoding check: does the CRF help, or does it smooth rare roles
# away (e.g. a precedent sentence inside a block of Ratio)?
# The better one on dev (CL + IT together) is used from here on
# and for the test predictions (CELL 14).
# ------------------------------------------------------------

print(f"{'decoding (dev macro-F1)':34s} {'CL':>7s} {'IT':>7s} {'all':>7s}")

for name, scores in [("CRF Viterbi (best role sequence)", crf_scores), ("best role of each sentence", plain_scores)]:
    print(f"{name:34s} {scores.get('CL', {}).get('f1', 0):7.4f} {scores.get('IT', {}).get('f1', 0):7.4f} {scores['all']['f1']:7.4f}")

if plain_scores["all"]["f1"] > crf_scores["all"]["f1"]:
    BEST_DECODING, dev_scores, dev_predictions = "no_crf", plain_scores, plain_predictions
else:
    BEST_DECODING, dev_scores, dev_predictions = "crf", crf_scores, crf_predictions

print(f"-> using: {BEST_DECODING}\n")

print(f"DEV ({RUN_NAME}, best epoch {best_epoch})")
print(f"{'':6s} {'macro-P':>8s} {'macro-R':>8s} {'macro-F1':>9s}")

for key in ["CL", "IT", "all"]:
    if key in dev_scores:
        s = dev_scores[key]
        print(f"{key:6s} {s['precision']:8.4f} {s['recall']:8.4f} {s['f1']:9.4f}")


# ------------------------------------------------------------
# Per label + most frequent mistakes
# ------------------------------------------------------------

per_label = {}

for domain in ["CL", "IT"]:

    gold, pred = flat_labels(dev_docs, dev_predictions, domain)

    if not gold:
        continue

    p, r, f, support = precision_recall_fscore_support(gold, pred, labels=LABELS, zero_division=0)

    per_label[domain] = {
        label: {"precision": float(p[k]), "recall": float(r[k]), "f1": float(f[k]), "support": int(support[k])}
        for k, label in enumerate(LABELS)
    }

    print(f"\n{domain}: {'label':24s} {'P':>6s} {'R':>6s} {'F1':>6s} {'gold':>5s}")

    for label in LABELS:
        s = per_label[domain][label]
        print(f"    {label:24s} {s['precision']:6.3f} {s['recall']:6.3f} {s['f1']:6.3f} {s['support']:5d}")

    confusions = Counter((g, q) for g, q in zip(gold, pred) if g != q)

    print(f"\n  most frequent mistakes in {domain} (gold -> predicted):")

    for (g, q), count in confusions.most_common(8):
        print(f"    {g:24s} -> {q:24s} {count:4d}")


# ------------------------------------------------------------
# Closely related roles
# ------------------------------------------------------------

RELATED_ROLES = {
    "precedents": ["PrecedentReliedUpon", "PrecedentNotReliedUpon", "PrecedentOverruled"],
    "arguments": ["ArgumentPetitioner", "ArgumentRespondent"],
    "decision": ["RatioOfTheDecision", "RulingByPresentCourt"],
    "background": ["Fact", "RulingByLowerCourt"],
}

print(f"\n{'group':12s} {'domain':6s} {'sentences':>9s} {'correct':>8s} {'confused within group':>22s}")

confusion_groups = {}

for group, roles in RELATED_ROLES.items():
    for domain in ["CL", "IT"]:

        gold, pred = flat_labels(dev_docs, dev_predictions, domain)
        pairs = [(g, q) for g, q in zip(gold, pred) if g in roles]

        if not pairs:
            continue

        correct = sum(g == q for g, q in pairs) / len(pairs)
        within = sum(g != q and q in roles for g, q in pairs) / len(pairs)

        confusion_groups[f"{group}_{domain}"] = {"sentences": len(pairs), "correct": correct, "within_group": within}

        print(f"{group:12s} {domain:6s} {len(pairs):9d} {correct:8.1%} {within:22.1%}")


# ------------------------------------------------------------
# Save
# ------------------------------------------------------------

results = {
    "run": RUN_NAME,
    "model": "Architecture-1 decoder (frozen) + IA3",
    "method": "ia3",
    "trained_on": "train only (dev used to pick the best epoch)",
    "parameters_total": n_total,
    "parameters_trainable": n_trainable,
    "parameters_adapter": n_adapter,
    "parameters_head": n_head,
    "best_epoch": best_epoch,
    "train_minutes": train_minutes,
    "peak_gpu_gb": peak_gpu_gb,
    "dev_predict_seconds": predict_seconds,
    "max_sentence_tokens": MAX_SENT_TOKENS,
    "window": WINDOW,
    "sentence_vector": "[mean ; max] of the token vectors",
    "best_epoch_selected_on": "dev macro-F1, CL + IT together",
    "decoding": {"crf": crf_scores, "no_crf": plain_scores, "used": BEST_DECODING},
    "dev": dev_scores,
    "dev_per_label": per_label,
    "dev_related_roles": confusion_groups,
    "never_predicted": {d: [l for l, ok in zip(LABELS, LABEL_ALLOWED[d]) if not ok] for d in ["CL", "IT"]},
    "history": history,
}

with open(OUTPUT_DIR / "results.json", "w", encoding="utf-8") as f:
    json.dump(results, f, indent=2)

with open(OUTPUT_DIR / "dev_prediction.json", "w", encoding="utf-8") as f:
    json.dump(dev_predictions, f, ensure_ascii=False, indent=1)

print("\nSaved:", OUTPUT_DIR / "results.json", "and dev_prediction.json")

decoding (dev macro-F1)                 CL      IT     all
CRF Viterbi (best role sequence)    0.3696  0.4507  0.4449
best role of each sentence          0.3558  0.4482  0.4284
-> using: crf

DEV (p3_arch1_ia3, best epoch 12)
        macro-P  macro-R  macro-F1
CL       0.3693   0.4357    0.3696
IT       0.5979   0.4335    0.4507
all      0.5569   0.4444    0.4449

CL: label                         P      R     F1  gold
    Fact                      0.487  0.586  0.532   249
    Issue                     0.000  0.000  0.000     3
    ArgumentPetitioner        0.641  0.436  0.519   344
    ArgumentRespondent        0.534  0.432  0.478   236
    Statute                   0.746  0.914  0.822    58
    Dissent                   0.000  0.000  0.000     0
    PrecedentReliedUpon       0.449  0.155  0.230   142
    PrecedentNotReliedUpon    0.000  0.000  0.000     0
    PrecedentOverruled        0.000  0.000  0.000     0
    RulingByLowerCourt        0.000  0.000  0.000    56
    RatioOfTheDec

In [20]:
# ============================================================
# CELL 14 — PREDICT TEST + WRITE prediction.json + submission.zip
# ============================================================

print("Decoding:", BEST_DECODING, "(chosen on dev in CELL 13)")
test_predictions = predict_labels(model, test_docs, use_crf=(BEST_DECODING == "crf"))


# ------------------------------------------------------------
# Format checks: one valid label per sentence, same order, every document
# ------------------------------------------------------------

ids = [doc["id"] for doc in test_docs]
assert len(ids) == len(set(ids)), "the same document id appears in CL and IT test files"

for doc in test_docs:

    labels = test_predictions[doc["id"]]

    assert len(labels) == len(doc["sentences"]), doc["id"]
    assert all(label in LABEL2ID for label in labels), doc["id"]

print(f"✅ {len(test_docs)} test documents, {sum(len(v) for v in test_predictions.values())} sentences, all checks passed")

print("\nPredicted label distribution (test):")
for label, count in Counter(l for v in test_predictions.values() for l in v).most_common():
    print(f"  {label:24s} {count:6d}")


PREDICTION_FILE = OUTPUT_DIR / "prediction.json"
ZIP_FILE = OUTPUT_DIR / "submission.zip"

with open(PREDICTION_FILE, "w", encoding="utf-8") as f:
    json.dump(test_predictions, f, ensure_ascii=False, indent=1)

# the same predictions split by domain (in case Codabench wants one file per domain)
for domain in ["CL", "IT"]:
    with open(OUTPUT_DIR / f"{domain}_prediction.json", "w", encoding="utf-8") as f:
        json.dump({d["id"]: test_predictions[d["id"]] for d in test_docs if d["domain"] == domain},
                  f, ensure_ascii=False, indent=1)

with zipfile.ZipFile(ZIP_FILE, "w", zipfile.ZIP_DEFLATED) as zf:
    zf.write(PREDICTION_FILE, arcname="prediction.json")        # file at the ROOT of the zip

with zipfile.ZipFile(ZIP_FILE) as zf:
    print("\nsubmission.zip:", zf.namelist())

print("Saved:", ZIP_FILE)

Decoding: crf (chosen on dev in CELL 13)
✅ 10 test documents, 2197 sentences, all checks passed

Predicted label distribution (test):
  RatioOfTheDecision          899
  Fact                        412
  PrecedentReliedUpon         288
  ArgumentPetitioner          262
  Statute                     156
  RulingByLowerCourt           69
  RulingByPresentCourt         54
  ArgumentRespondent           28
  None                         21
  Issue                         5
  PrecedentNotReliedUpon        3

submission.zip: ['prediction.json']
Saved: /kaggle/working/p3_arch1_ia3/submission.zip
